# Solution 02 — Multi-hop Ops QA

Reference implementation for Assessment 02.


## Architectural Decisions

Pipeline: **reformulate → select evidence → answer → policy gate**

1. **Reformulate** the question into sub-questions / info needs (helps multi-hop).
2. **Select** relevant KB ids given those needs (grounding / routing over snippets).
3. **Answer** strictly from selected snippets with citations and confidence.
4. **Customer-safety rewrite** using messaging policy snippets.

This differs from Assessment 01's document-critique pattern: here the hard part is **retrieval planning + grounded synthesis + audience split**.


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
KNOWLEDGE_BASE = [
  {
    "id": "KB-01",
    "title": "Payment Service SLOs",
    "text": "The Payments API targets 99.9% monthly availability. Error budget burn >2% in 1 hour triggers a SEV-2. Dependency: Ledger Service and Card Network Gateway."
  },
  {
    "id": "KB-02",
    "title": "Ledger Service",
    "text": "Ledger Service stores immutable transaction journals. It runs in active-active across us-east-1 and us-west-2. Schema migrations require a dual-write period of 24 hours."
  },
  {
    "id": "KB-03",
    "title": "Card Network Gateway",
    "text": "Card Network Gateway talks to Visa/MC. Nightly maintenance window: 01:00\u201303:00 UTC on Sundays. During maintenance, authorization latency can spike to 2s; the Payments API retries twice with jitter."
  },
  {
    "id": "KB-04",
    "title": "Incident Severity Guide",
    "text": "SEV-1: complete payment outage or data loss. SEV-2: material SLO burn or partial regional outage. SEV-3: degraded performance without SLO breach. On-call for Payments owns SEV-1/2; Platform owns SEV-3 unless escalated."
  },
  {
    "id": "KB-05",
    "title": "Customer Messaging Policy",
    "text": "For SEV-1/2 payment incidents lasting >15 minutes, Status Page must be updated. Customer Support may say 'payment processing delays' but must not name vendors (Visa/MC) or internal service names."
  },
  {
    "id": "KB-06",
    "title": "Retry & Idempotency",
    "text": "Clients must send Idempotency-Key on POST /charges. Duplicate keys within 24h return the original response. Retries without keys may double-charge; Finance runbook FR-17 handles remediation."
  },
  {
    "id": "KB-07",
    "title": "us-east-1 Capacity Event (historical)",
    "text": "In March, a us-east-1 capacity event caused Ledger failover to us-west-2. Payments API stayed up but p99 latency rose to 900ms for 40 minutes. Classified SEV-2. Status Page updated at T+12 minutes."
  },
  {
    "id": "KB-08",
    "title": "Support Escalation",
    "text": "If >50 customers report charge failures in 10 minutes, Support pages Payments on-call even without monitoring alerts. Use bridge channel #pay-incident."
  }
]

EXAMPLE_QUESTION = (
    "Sunday 02:10 UTC we see payment authorization latency around 2 seconds and elevated "
    "customer complaints. What severity should we consider, who owns it, and what can "
    "Support tell customers if this lasts 20 minutes?"
)


## Why Each Call

| Call | Necessity |
|------|-----------|
| Reformulate | Turns compound questions into hops the selector can match |
| Select | Reduces distraction from irrelevant KB rows; improves citation precision |
| Answer | Produces engineer-facing grounded answer |
| Safety gate | Enforces customer messaging constraints as a separate concern |


In [ ]:
def answer_ops_question(question: str, knowledge_base: list) -> dict:
    if not question or not knowledge_base:
        return {
            "answer": "No question or knowledge base provided.",
            "citations": [],
            "reasoning_trace": [],
            "confidence": "low",
            "missing_info": ["question or KB"],
            "safe_for_customer_support": False,
            "customer_safe_rephrase": None,
        }

    kb_index = {
        row["id"]: f"{row['id']} | {row['title']}: {row['text']}"
        for row in knowledge_base
    }
    catalog = "\n".join(kb_index.values())

    # Call 1: reformulate into sub-questions
    plan_raw = call_llm(
        prompt=(
            "Break the user question into 2-5 concrete sub-questions that must be answered "
            "from an internal ops knowledge base. Return JSON "
            "{\"subquestions\":[...], \"notes\":\"...\"} only.\n\nQUESTION:\n" + question
        ),
        system_prompt="You plan multi-hop retrieval for ops Q&A.",
        temperature=0.1,
    )
    try:
        plan = extract_json(plan_raw)
        subqs = plan.get("subquestions", [])
    except ValueError:
        subqs = [question]

    # Call 2: select KB ids
    select_raw = call_llm(
        prompt=(
            "Given sub-questions and KB catalog, select the minimal set of KB ids needed. "
            "Return JSON {\"selected_ids\":[...], \"rationale\":\"...\"} only.\n\n"
            f"SUBQUESTIONS:\n{json.dumps(subqs)}\n\nCATALOG:\n{catalog}"
        ),
        system_prompt="You select evidence snippets. Prefer precision over recall.",
        temperature=0.0,
    )
    try:
        selected = extract_json(select_raw).get("selected_ids", [])
    except ValueError:
        selected = list(kb_index.keys())[:4]

    selected = [i for i in selected if i in kb_index]
    if not selected:
        selected = list(kb_index.keys())[:3]
    evidence = "\n".join(kb_index[i] for i in selected)

    # Call 3: grounded answer
    ans_raw = call_llm(
        prompt=(
            "Answer the engineer question using ONLY the evidence. Return JSON with keys: "
            "answer, citations (subset of evidence ids), reasoning_trace (list of strings), "
            "confidence (high|medium|low), missing_info (list).\n\n"
            f"QUESTION:\n{question}\n\nEVIDENCE:\n{evidence}"
        ),
        system_prompt="Grounded ops assistant. Never invent runbooks or SLOs.",
        temperature=0.1,
    )
    try:
        ans = extract_json(ans_raw)
    except ValueError:
        ans = {
            "answer": ans_raw,
            "citations": selected,
            "reasoning_trace": subqs,
            "confidence": "low",
            "missing_info": ["parse_failure"],
        }

    citations = [c for c in ans.get("citations", selected) if c in kb_index]

    # Call 4: customer safety gate
    policy_bits = "\n".join(
        kb_index[i] for i in kb_index if i in citations or "Customer" in kb_index[i] or i == "KB-05"
    )
    gate_raw = call_llm(
        prompt=(
            "Determine if the engineer answer is safe to send to external customers per policy. "
            "Return JSON {\"safe_for_customer_support\": bool, "
            "\"customer_safe_rephrase\": string or null, \"reason\":\"...\"}\n\n"
            f"ENGINEER_ANSWER:\n{ans.get('answer','')}\n\nPOLICY/EVIDENCE:\n{policy_bits}"
        ),
        system_prompt="You enforce customer messaging policy.",
        temperature=0.0,
    )
    try:
        gate = extract_json(gate_raw)
    except ValueError:
        gate = {"safe_for_customer_support": False, "customer_safe_rephrase": None}

    return {
        "answer": ans.get("answer", ""),
        "citations": citations,
        "reasoning_trace": ans.get("reasoning_trace", subqs),
        "confidence": ans.get("confidence", "medium"),
        "missing_info": ans.get("missing_info", []),
        "safe_for_customer_support": bool(gate.get("safe_for_customer_support", False)),
        "customer_safe_rephrase": gate.get("customer_safe_rephrase"),
    }


## Example Execution


In [ ]:
result = answer_ops_question(EXAMPLE_QUESTION, KNOWLEDGE_BASE)
print(json.dumps(result, indent=2)[:3500])


## Alternatives

- Embeddings + top-k then one LLM call: better at scale; overkill for 8 snippets.
- Full-KB single call: simplest; weaker multi-hop and citation discipline.
- Agent loop with tools: flexible but easy to burn the time box.

## Tradeoffs

Four calls add latency; selection call is the main quality lever. Lower temperature on select/gate.

## Common Mistakes

- Dumping entire KB into one prompt and hoping.
- Citations that were never read.
- Customer-unsafe answers (naming Visa/MC or internal services).

## Beyond 45 Minutes

- Hybrid BM25/embeddings; refusal calibration; eval set of multi-hop questions.
